# Single URL product scrape

Crawl4AI-only agentic scraper with multi-profile capture, strict capture decisions, source alignment, quality gate, and product-only artifacts.


In [ ]:
from pathlib import Path
import sys, json

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_PATH:", SRC_PATH)

## Optional: reset package cache after replacing code
Run this cell after copying a new zip into the same notebook kernel.

In [ ]:
import sys
for name in list(sys.modules):
    if name.startswith("product_scraping_agent"):
        del sys.modules[name]
print("product_scraping_agent modules cleared. Re-run imports below.")

In [ ]:
from product_scraping_agent import ProductScrapingAgent, ScrapeRequest
from product_scraping_agent.models import ScrapeResult
print("Import OK")
print("ScrapeResult has capture_score:", "capture_score" in ScrapeResult.model_fields)

## Build request
`product_url` is the actual URL to scrape. Requested retailer/country and source retailer/country are separated so fallback URLs are handled safely.

In [ ]:
request = ScrapeRequest(
    product_url="https://www.amazon.com/Barbie-Collectible-All-Denim-Signature-Underwear/dp/B0BHFD7BPM",
    main_text="THE MOVIE 2023 KEN DOLL WEARING DENIM SET",
    ean="194735174539",

    # Original business target context
    requested_retailer_name="Mercado Libre",
    requested_country_code="CO",

    # Actual URL/source context
    source_retailer_name="Amazon",
    source_country_code="US",
    source_url_role="global_fallback",

    output_root=PROJECT_ROOT / "data" / "scraped",
    max_images=30,
    vision_max=12,
    max_agent_iterations=2,
    write_raw_debug=False,
)
request

## Run scrape

In [ ]:
agent = ProductScrapingAgent(output_root=PROJECT_ROOT / "data" / "scraped")
result = await agent.scrape(request)
result

## Result summary

In [ ]:
print("Success:", result.success)
print("Quality:", result.artifact_quality)
print("Quality score:", result.quality_score)
print("Manual review:", result.requires_manual_review)
print("Output dir:", result.output_dir)
print("Access:", result.access_status, result.access_issue_type)
print("Browser visible:", result.browser_visible)
print("Capture profile used:", result.capture_profile_used)
print("Profiles attempted:", result.capture_profiles_attempted)
print("Capture score:", result.capture_score)
print("Capture grade:", result.capture_grade)
print("Capture decision:", result.capture_decision)
print("Real scrape evidence:", result.real_scrape_evidence)
print("Weak reasons:", result.weak_capture_reasons)
print("Evidence axes:", result.evidence_axes_used)
print("Images final/candidates:", result.final_image_count, result.image_candidate_count)
print("Tables:", result.table_count, "JSON-LD:", result.json_ld_count)
print("Error:", result.error)

## Read reports

In [ ]:
def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path and path.exists() else {}

quality = read_json(result.quality_report_path)
alignment = read_json(result.source_alignment_report_path)
metadata = read_json(result.metadata_json_path)

print("QUALITY REPORT")
print(json.dumps(quality, ensure_ascii=False, indent=2)[:4000])
print("\nSOURCE ALIGNMENT REPORT")
print(json.dumps(alignment, ensure_ascii=False, indent=2)[:4000])
print("\nCAPTURE PROFILE SCORES")
print(json.dumps(metadata.get("capture_profile_scores", []), ensure_ascii=False, indent=2)[:4000])

## Inspect final image folder
Only final retained product images should be under `retailer/images/`. Rejected candidates stay in `image_manifest.json`.

In [ ]:
img_dir = Path(result.output_dir) / "images"
print("Image dir:", img_dir)
print("Exists:", img_dir.exists())
print("Files:", sorted([p.name for p in img_dir.iterdir()]) if img_dir.exists() else [])

image_manifest = read_json(result.image_manifest_path)
print("Manifest rows:", len(image_manifest))
print(json.dumps(image_manifest[:3], ensure_ascii=False, indent=2)[:3000])

## Open Markdown artifacts

In [ ]:
for label, path in [
    ("product_evidence.md", result.product_evidence_md_path),
    ("claims.md", result.claims_md_path),
    ("vision.md", result.vision_md_path),
]:
    print("\n" + "="*80)
    print(label, path)
    print("="*80)
    p = Path(path)
    print(p.read_text(encoding="utf-8")[:5000] if p.exists() else "missing")